# 🔮 Bidirectional LSTM (BiLSTM) Network
**Sequence Classification with Keras/TensorFlow**
---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

print(f'TensorFlow version : {tf.__version__}')
print('Libraries loaded ✅')

## 2. Load & Explore Dataset
> Using a **Synthetic Time Series** dataset. Instead of forecasting the next value, we will frame this as a **Sequence Classification** task: determining if a given sliding window exhibits a net *Upward* or *Downward* trajectory.

In [ ]:
# Generate dataset
np.random.seed(42)
t = np.linspace(0, 100, 2000)
y = 3 * np.sin(0.1 * t) + 1.5 * np.cos(0.3 * t) + 0.02 * t + np.random.randn(2000) * 0.8
df = pd.DataFrame({'time': t, 'value': y})

print(f'Shape   : {df.shape}')
df.head()

## 3. Time Series Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['time'], df['value'], color='#8b5cf6', lw=1.5, label='Value')
ax.set_title('Synthetic Time Series (Trend + Seasonality + Noise)', fontsize=14, fontweight='bold')
ax.set_xlabel('Time'); ax.set_ylabel('Value')
ax.legend()
plt.tight_layout(); plt.show()

## 4. Data Preprocessing for Classification
> We create sliding windows. For each window, the **label is 1** if the value at the end of the window is greater than the value at the start (Upward trend), and **0** otherwise (Downward/Flat). This is a perfect use case for BiLSTM, as the model can look at the entire window to make the decision.

In [ ]:
def create_sequences_with_labels(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        window = data[i:i + seq_length]
        X.append(window)
        # Label: 1 if end > start (Upward), else 0
        label = 1 if window[-1] > window[0] else 0
        y.append(label)
    return np.array(X), np.array(y)

SEQ_LENGTH = 30
values = df['value'].values.reshape(-1, 1)

scaler = StandardScaler()
scaled_values = scaler.fit_transform(values).flatten()

X, y = create_sequences_with_labels(scaled_values, SEQ_LENGTH)
X = np.reshape(X, (X.shape[0], X.shape[1], 1))

# Train/test split
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'X_test shape : {X_test.shape}')
print(f'Class distribution (Train): {np.bincount(y_train)}')

## 5. What makes BiLSTM special?
> A standard LSTM processes data strictly from past to future ($x_1 \rightarrow x_T$). A **Bidirectional LSTM** runs two independent LSTMs:
1. **Forward LSTM**: Processes $x_1 \rightarrow x_T$ (captures past context).
2. **Backward LSTM**: Processes $x_T \rightarrow x_1$ (captures future context).

At each time step, their hidden states are concatenated: $h_t = [\overrightarrow{h_t}, \overleftarrow{h_t}]$. This doubles the representational capacity and is ideal for tasks where the full sequence is known at inference time (e.g., classification, smoothing, NLP).

## 6. Build BiLSTM Model

In [ ]:
def build_bilstm(units=32, dropout=0.2, seq_length=30, layers=1):
    model = keras.Sequential(name='BiLSTM_Model')
    
    for i in range(layers):
        return_seq = True if i < layers - 1 else False
        input_shape = (seq_length, 1) if i == 0 else None
        
        # Bidirectional wrapper applies LSTM in both directions and concatenates
        model.add(layers.Bidirectional(
            layers.LSTM(units, return_sequences=return_seq),
            input_shape=input_shape
        ))
        if i < layers - 1:
            model.add(layers.Dropout(dropout))
            
    model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation='sigmoid')) # Binary classification
    
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])
    return model

model = build_bilstm(units=32, dropout=0.2, seq_length=SEQ_LENGTH, layers=1)
model.summary()

## 7. Train the Model

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True, mode='max', verbose=1)
]

start_time = time.time()
history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)
print(f'Training completed in {time.time() - start_time:.2f} seconds')

## 8. Training History

In [ ]:
hist = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist['loss'], color='#e05252', lw=2, label='Train')
axes[0].plot(hist['val_loss'], color='#8b5cf6', lw=2, label='Val')
axes[0].set_title('Training Loss (Binary Crossentropy)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(hist['accuracy'], color='#34d399', lw=2, label='Train')
axes[1].plot(hist['val_accuracy'], color='#f59e0b', lw=2, label='Val')
axes[1].set_title('Training Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout(); plt.show()

## 9. Evaluate on Test Set

In [ ]:
y_pred_prob = model.predict(X_test).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print('='*50)
print('          BiLSTM Test Set Results')
print('='*50)
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print('='*50)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Downward (0)', 'Upward (1)']))

## 10. Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[0],
            xticklabels=['Downward (0)', 'Upward (1)'],
            yticklabels=['Downward (0)', 'Upward (1)'],
            linewidths=1, linecolor='white')
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#8b5cf6', lw=2.5, label=f'BiLSTM (AUC = {roc_auc:.4f})')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#8b5cf6')
axes[1].plot([0,1],[0,1],'k--', lw=1.5, label='Random Classifier')
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout(); plt.show()

## 11. BiLSTM vs Standard LSTM Comparison

In [ ]:
# Compare BiLSTM and standard LSTM on this classification task
results = {}

for model_type in ['LSTM', 'BiLSTM']:
    print(f'Training {model_type}...')
    if model_type == 'LSTM':
        m = keras.Sequential([
            layers.LSTM(32, input_shape=(SEQ_LENGTH, 1)), # 32 units total
            layers.Dropout(0.2),
            layers.Dense(1, activation='sigmoid')
        ])
    else:
        m = build_bilstm(units=32, dropout=0.2, seq_length=SEQ_LENGTH, layers=1) # 32 units per direction (64 total)
    
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    cb = [EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True, mode='max')]
    
    start = time.time()
    m.fit(X_train, y_train, validation_split=0.15, epochs=80, batch_size=32, callbacks=cb, verbose=0)
    train_time = time.time() - start
    
    yp_prob = m.predict(X_test).flatten()
    yp = (yp_prob >= 0.5).astype(int)
    acc = accuracy_score(y_test, yp)
    params = m.count_params()
    
    results[model_type] = {'Accuracy': acc, 'Time (s)': train_time, 'Params': params}
    print(f'  -> Acc: {acc:.4f}, Time: {train_time:.2f}s, Params: {params}')

res_df = pd.DataFrame(results).T
print('\nComparison Summary:')
display(res_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(res_df.index, res_df['Accuracy'], color=['#34d399', '#8b5cf6'], edgecolor='white')
axes[0].set_title('Test Accuracy (Higher is Better)', fontsize=13, fontweight='bold')
axes[0].set_ylim(0.5, 1.0)

axes[1].bar(res_df.index, res_df['Time (s)'], color=['#34d399', '#8b5cf6'], edgecolor='white')
axes[1].set_title('Training Time', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Seconds')
plt.tight_layout(); plt.show()

## 12. Save Model & Scaler

In [ ]:
import os, joblib
os.makedirs('../models', exist_ok=True)
model.save('../models/bilstm_model.keras')
joblib.dump(scaler, '../models/scaler.pkl')
print('Model saved  → models/bilstm_model.keras')
print('Scaler saved → models/scaler.pkl')

## 13. Key Takeaways
> - **BiLSTMs excel at Sequence Classification**: When the entire input window is available, processing it in both directions provides richer context than a unidirectional LSTM.
> - **Not for Forecasting**: BiLSTMs cannot be used for real-time future prediction because the backward pass requires future data that does not yet exist.
> - **Parameter Cost**: A BiLSTM with $N$ units has roughly twice the parameters of a standard LSTM with $N$ units, leading to longer training times, but often yielding higher accuracy on classification tasks.
> - **Merge Modes**: While 'concat' is the default and most common, Keras also supports 'sum', 'mul', and 'ave' for merging the forward and backward states.